In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score,GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error,mean_squared_error

In [3]:
df = pd.read_csv('cleaned_engineered.csv')

In [4]:
skew_num = ['Study_Hours']
other_num = ['Age','Avg_Daily_Usage_Hours','Daily_Unlocks','Physical_Activity_Hours','Sleep_Hours_Per_Night']
ord_cat = ['Stress_Level','Academic_Level']
ohe_cat = ['Gender','Country','Most_Used_Platform','Purpose_Of_Use']
cols = skew_num+other_num+ord_cat+ohe_cat
X = df[cols]
y = df['Mental_Health_Score']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
#1. Skewed features
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())

])

#2. Numeric Features
plain_numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

#3. Ordinal
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High'],['High School','Undergraduate','Graduate',]]))
])

#4. Nominal Features
nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore",drop='first'))
])


preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skew_num),
    ("Plain_Numeric",plain_numeric_pipeline, other_num ),
    ('Ordinal', ordinal_pipeline, ord_cat),
    ('Normal', nominal_pipeline, ohe_cat)
])


In [6]:
xg_pipe = Pipeline([['preprocesser',preprocessor],['xgb',XGBRegressor()]])

In [7]:
xg_pipe.fit(X_train,y_train)

,steps,"[('preprocesser', ...), ['xgb', XGBRegressor(...ree=None, ...)]]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
y_pred_test = xg_pipe.predict(X_test)
y_pred_train = xg_pipe.predict(X_train)

In [10]:
print("Test r2",r2_score(y_test,y_pred_test))
print("Train r2",r2_score(y_train,y_pred_train))
print ("Test MAE", mean_absolute_error(y_test,y_pred_test))
print("Test RMSE",root_mean_squared_error(y_test,y_pred_test))

Test r2 0.908332884154724
Train r2 0.971709584924349
Test MAE 0.2975161345481872
Test RMSE 0.404530447816542


In [11]:
scores = cross_val_score(xg_pipe,X,y,cv=5,scoring='r2')

In [12]:
scores.mean()

np.float64(0.9101390176991451)

In [21]:
params = {'xgb__n_estimators' : [500,700,900],'xgb__learning_rate' : [0.01,0.1,0.3,0.5],'xgb__max_depth' : [10,13,15]
         ,'xgb__min_child_weight' : [1,3,5],'xgb__colsample_bytree' : [0.75,1]}

In [22]:
grid = GridSearchCV(xg_pipe,param_grid=params,cv=3,n_jobs=-1,scoring='r2',verbose=3)

In [23]:
grid.fit(X_train,y_train)

Fitting 3 folds for each of 216 candidates, totalling 648 fits

[CV 3/3] END xgb__colsample_bytree=1, xgb__learning_rate=0.01, xgb__max_depth=5, xgb__min_child_weight=5, xgb__n_estimators=500, xgb__subsample=1.0;, score=0.846 total time=   0.3s
[CV 1/3] END xgb__colsample_bytree=1, xgb__learning_rate=0.01, xgb__max_depth=7, xgb__min_child_weight=1, xgb__n_estimators=300, xgb__subsample=1.0;, score=0.858 total time=   0.4s
[CV 1/3] END xgb__colsample_bytree=1, xgb__learning_rate=0.01, xgb__max_depth=7, xgb__min_child_weight=3, xgb__n_estimators=100, xgb__subsample=0.75;, score=0.705 total time=   0.1s
[CV 3/3] END xgb__colsample_bytree=1, xgb__learning_rate=0.01, xgb__max_depth=7, xgb__min_child_weight=3, xgb__n_estimators=100, xgb__subsample=0.75;, score=0.716 total time=   0.1s
[CV 1/3] END xgb__colsample_bytree=1, xgb__learning_rate=0.01, xgb__max_depth=7, xgb__min_child_weight=3, xgb__n_estimators=300, xgb__subsample=0.75;, score=0.855 total time=   0.3s
[CV 3/3] END xgb__colsample_

,estimator,"Pipeline(step...=None, ...)]])"
,param_grid,"{'xgb__colsample_bytree': [0.75, 1], 'xgb__learning_rate': [0.01, 0.1, ...], 'xgb__max_depth': [10, 13, ...], 'xgb__min_child_weight': [1, 3, ...], ...}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"


In [24]:
grid.best_score_

np.float64(0.8978105146043759)

In [25]:
grid.best_params_

{'xgb__colsample_bytree': 0.75,
 'xgb__learning_rate': 0.1,
 'xgb__max_depth': 10,
 'xgb__min_child_weight': 1,
 'xgb__n_estimators': 900}

## XGBoost Regression – Model Evaluation

XGBoost Regression was implemented to further investigate the non-linear relationship between the input features and the Mental Health Score. XGBoost is a gradient boosting algorithm that builds decision trees sequentially, where each new tree attempts to reduce the errors made by the previous trees.

A baseline XGBoost model was first trained and evaluated before performing hyperparameter tuning.

The baseline model achieved a mean **5-fold cross-validation R² score of approximately 0.9101**. On the held-out test set, the model achieved a **Test R² of approximately 0.90**, while the training R² was approximately **0.97**.

The relatively high training R² compared with the test R² indicates that the model fits the training data very strongly and exhibits some degree of overfitting. However, the test R² and cross-validation R² remain high, indicating that the model generalizes well to unseen observations.

### Hyperparameter Tuning

GridSearchCV with 5-fold cross-validation was subsequently used to search for better combinations of XGBoost hyperparameters.

The best configuration found during the search achieved a cross-validation R² score of approximately **0.89**.

Interestingly, the tuned model did not outperform the baseline model, whose cross-validation R² was approximately **0.9101**.

This demonstrates that hyperparameter tuning does not necessarily guarantee an improvement in model performance. GridSearchCV only identifies the best-performing configuration among the hyperparameter combinations included in the search space. If the baseline configuration is not included or the searched configurations are more restrictive, the tuned model may perform worse than the original baseline.

### Results

| Model | 5-Fold CV R² | Test R² | Train R² |
|---|---:|---:|---:|
| Baseline XGBoost | **0.9101** | **≈ 0.90** | **≈ 0.97** |
| Tuned XGBoost | ≈ 0.89 | — | — |

### Conclusion

The baseline XGBoost model achieved better cross-validation performance than the hyperparameter-tuned configurations explored. Therefore, the baseline XGBoost model is retained as the preferred XGBoost configuration for this dataset.

The baseline model's high training R² compared with its test R² suggests some overfitting; however, its strong cross-validation and test performance indicate good predictive capability.

The baseline XGBoost model will be compared with the previously evaluated Linear Regression, Ridge, Lasso, SVR, and Random Forest models to determine the overall best-performing regression algorithm.